In [ ]:
# Module1_DemoLab_LinearRegression.ipynb

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (StandardScaler, PolynomialFeatures)
from scipy.stats import boxcox
from scipy.special import boxcox as bc_fixed
from scipy.special import inv_boxcox
import pandas as pd

# read dataset
boston = pd.read_pickle("data/boston_housing_clean.pickle")
boston_data = boston['dataframe']

lr = LinearRegression()
# segregate x and y data
X = boston_data.drop('MEDV', axis=1) # drop the target column
y = boston_data['MEDV']

# create PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
X_pf = pf.fit_transform(X)  # transform the features data for polynomial features on the X dataset

# extract training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X_pf, y, test_size=0.3, random_state=72018)

# scale the training features data
s = StandardScaler()
X_train_s = s.fit_transform(X_train)

bc_result2 = boxcox(y_train) # transform the training target data using boxcox()
y_train_bc = bc_result2[0]  # the transformed target variable
lam2 = bc_result2[1]        # the lambda: power used for the transformation.

lr.fit(X_train_s, y_train_bc) # train model with the training dataset

X_test_s = s.transform(X_test)    # why do we use transform() instead of fit_transform() ?
y_pred_bc = lr.predict(X_test_s)  # generate the predicted result with the above transformed testing data

# Use the transformed test target and the predictions (which are also on the transformed scale)
y_test_bc = bc_fixed(y_test, lam2)
r2_transformed = r2_score(y_test_bc, y_pred_bc)
print(f"R2 in Transformed Space: {r2_transformed:.4f}")

y_pred_original = inv_boxcox(y_pred_bc, lam2)
display(y_pred_original[:10])
display(y_test.values[:10])

# R-square: compare to the original, untransformed y_test
r2_original = r2_score(y_test, y_pred_original)
print(f"R2 in Original Space: {r2_original:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline


df = pd.read_csv("data/cleaned_hotEncoded_car_data.csv")

# allocate training and testing data
X_train, X_test, y_train, y_test = train_test_split( df, y, test_size=0.30, random_state=42)

# scale the training features
ss = StandardScaler()
X_train_s = ss.fit_transform(X_train)



# train the model
lm = LinearRegression()
lm.fit(X_train_s,y_train)

# scale the testing features
X_test_s = ss.transform(X_test)

# predict with test data
car_price_predictions = lm.predict(X_test_s)

# verify predictions
print("Verify predictions...")
display(f"mean squared error: {mean_squared_error(y_test, car_price_predictions)}")
display(f"lm score: {lm.score(X_test_s,y_test)}")
display(f"r2 score: {r2_score(y_test,car_price_predictions)}")


# using pipeline
steps=[('scaler', StandardScaler()), ('lm',  LinearRegression())]
pipe = Pipeline(steps=steps)
pipe.fit(X_train,y_train)
car_price_predictions = pipe.predict(X_test)

# verify predictions from pipeline
print("\n\nVerify Pipeline predictions...")
mse = mean_squared_error(y_test, car_price_predictions)
display(f"mean squared error: {mse}")
display(f"rmse: {np.sqrt(mse)}")
display(f"r2 score: {r2_score(y_test, car_price_predictions)}")





In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

 
data = pd.read_csv("data/Ames_Housing_Sales.csv")

## Determine potential number of additional columns when performing One-Hot-Encoding

numeric_cols     = data.select_dtypes(include=['number']).columns
categorical_cols = data.select_dtypes(exclude=['number']).columns
# Determine how many extra columns would be created when performing one-hot-encoding
num_ohc_cols = data[categorical_cols].apply(lambda x: x.nunique(dropna = False)).sort_values(ascending=False)

# No need to encode if there is only one value
small_num_ohc_cols = num_ohc_cols.loc[num_ohc_cols>1]

# Number of one-hot columns is one less than the number of categories
small_num_ohc_cols -= 1


## One Hot Encoding

# Copy of the data
# data_ohc = data.copy()
data_ohc = data.drop(numeric_cols, axis=1)

# The encoders
le = LabelEncoder()
ohc = OneHotEncoder()

for col in small_num_ohc_cols.index:
    # Integer encode the string categories
    dat = le.fit_transform(data_ohc[col]).astype(int)
    
    # Remove the original column from the dataframe
    data_ohc = data_ohc.drop(col, axis=1)

    # One hot encode the data--this returns a sparse array
    new_dat = ohc.fit_transform(dat.reshape(-1,1))

    # Create unique column names
    n_cols = new_dat.shape[1]
    col_names = ['_'.join([col, str(x)]) for x in range(n_cols)]

    # Create the new dataframe
    new_df = pd.DataFrame(new_dat.toarray(), index=data_ohc.index, columns=col_names)

    # Accumulate the new data to the dataframe
    data_ohc = pd.concat([data_ohc, new_df], axis=1)

# Remove the original string columns from the dataframe
data = data.drop(num_ohc_cols.index, axis=1)

print(f"data.shape: {data.shape}")
print(f"data_ohc.shape: {data_ohc.shape}")


In [7]:
## A better way to do the same for the above.

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

data = pd.read_csv("data/Ames_Housing_Sales.csv")
categorical_cols = data.select_dtypes(exclude=['number']).columns

# Determine how many extra columns would be created when performing one-hot-encoding
num_ohc_cols = data[categorical_cols].apply(lambda x: x.nunique(dropna = False)).sort_values(ascending=False)

data_ohc = data.copy()

# 1. Initialize the encoder
# sparse_output=False makes it return a normal array instead of a "sparse" one
# handle_unknown='ignore' prevents crashes if new data has a weird category
ohc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Apply it directly to your string columns
# OneHotEncoder will "learn" the strings and "transform" them into 1s and 0s in one shot
ohc_data = ohc.fit_transform(data[categorical_cols])

# 3. Get the column names automatically!
# This replaces that manual '_'.join() list comprehension
new_col_names = ohc.get_feature_names_out(categorical_cols)

# 4. Create the final DataFrame
ohe_df = pd.DataFrame(ohc_data, columns=new_col_names, index=data.index)

# 5. COMBINE the numerical columns (from 'data') with the new OHE columns
# We use pd.concat to glue the numerical features and the categorical features together
numerical_data = data.select_dtypes(include=['number'])
data_ohc = pd.concat([numerical_data, ohe_df], axis=1)

# Remove the original string columns from the dataframe
data = data.drop(num_ohc_cols.index, axis=1)




from sklearn.model_selection import train_test_split

y_col = 'SalePrice'

# Split the data that is not one-hot encoded
feature_cols = [x for x in data.columns if x != y_col]
X_data = data[feature_cols]
y_data = data[y_col]
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)

# Split the data that is one-hot encoded
feature_cols = [x for x in data_ohc.columns if x != y_col]
X_data_ohc = data_ohc[feature_cols]
y_data_ohc = data_ohc[y_col]
X_train_ohc, X_test_ohc, y_train_ohc, y_test_ohc = train_test_split(X_data_ohc, y_data_ohc, test_size=0.3, random_state=42)

# Compare the indices to ensure they are identical
# (X_train_ohc.index == X_train.index).all()
print(f"X_train_ohc shape: {X_train_ohc.shape}")
print(f"X_train shape: {X_train.shape}")


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

LR_no_enc = LinearRegression()
LR_ohc = LinearRegression()

# Storage for error values
error_df = list()

# Data that have not been one-hot encoded
LR_no_enc = LR_no_enc.fit(X_train, y_train)
y_train_pred = LR_no_enc.predict(X_train)
y_test_pred = LR_no_enc.predict(X_test)

error_df.append(pd.Series({'train': mean_squared_error(y_train, y_train_pred),
                           'test' : mean_squared_error(y_test,  y_test_pred)},
                           name='no enc'))


# Data that have been one-hot encoded
LR_ohc = LR_ohc.fit(X_train_ohc, y_train_ohc)
y_train_ohc_pred = LR_ohc.predict(X_train_ohc)
y_test_ohc_pred = LR_ohc.predict(X_test_ohc)

error_df.append(pd.Series({'train': mean_squared_error(y_train_ohc, y_train_ohc_pred),
                           'test' : mean_squared_error(y_test_ohc,  y_test_ohc_pred)},
                          name='one-hot enc'))

# Assemble the results
error_df = pd.concat(error_df, axis=1)
error_df

X_train_ohc shape: (965, 294)
X_train shape: (965, 36)


,no enc,one-hot enc
train,1.131507e+09,3.177267e+08
test,1.372182e+09,8.065328e+09


In [8]:
print(f"No Enc R2: {LR_no_enc.score(X_test, y_test)}")
print(f"OHC R2: {LR_ohc.score(X_test_ohc, y_test_ohc)}")

No Enc R2: 0.7727012498678184
OHC R2: -0.33600237067288163
